# notebooks/04_generation_test.ipynb

In [1]:
%run notebook_init.py

In [2]:
from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.verifier.hallucination_guard import sentence_overlap
from rag_pipeline.llm.generator import generate_answer, format_prompt, generate_answer_with_gate

/Users/aditya/Documents/projects/hallucination-resistant-finance-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dryrun = True # Set to TRUE by default

In [ ]:
def ask_and_verify():
    while True:
        q = input("Ask a question (or 'q' to quit): ")
        if q.lower() == 'q':
            break
        chunks = query_chunks(q)

        print("\n🔎 Retrieved Chunks (Debug Info):")
        for i, c in enumerate(chunks):
            metadata = c.get('metadata', {})
            print(f"\n  Chunk {i+1}:")
            print(f"    doc_id: {metadata.get('doc_id', 'unknown')}")
            print(f"    chunk_id: {metadata.get('chunk_id', 'unknown')}")
            print(f"    section: {metadata.get('section', 'unknown')}")
            print(f"    distance: {c.get('distance', 'N/A')}")
            print(f"    text preview: {c['text'][:200]}...")

        answer, abstained = generate_answer_with_gate(q, chunks, model_name="gpt-4o")
        print("\nAnswer:\n", answer)
        if abstained:
            print("(System abstained from answering)")

In [ ]:
ask_and_verify()

In [ ]:
# Use query_chunks() as single source of truth for retrieval
# This ensures consistent metadata and avoids mismatched Chroma instances

for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"doc_id: {c['metadata'].get('doc_id', 'unknown')}")
    print(f"chunk_id: {c['metadata'].get('chunk_id', 'unknown')}")
    print(f"section: {c['metadata'].get('section', 'unknown')}")
    print(f"distance: {c.get('distance', 'N/A')}")
    print(f"text preview: {c['text'][:300]}...")

# Debug Code Below

In [3]:
from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.llm.generator import generate_answer_with_gate

chunks = query_chunks("Where is Tesla headquartered?", top_k=5)
answer, abstained = generate_answer_with_gate(
    "Where is Tesla headquartered?",
    chunks
)
print(f"Answer: {answer}")
print(f"Abstained: {abstained}")

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/paraphrase-MiniLM-L3-v2


🔍 Loaded embedding model: sentence-transformers/paraphrase-MiniLM-L3-v2


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]


DEBUG: keywords=['tesla', 'headquartered']
DEBUG: keywords=['tesla', 'headquartered']
DEBUG: use_keyword_fallback=True, keyword_matches_full count=232
  Keyword match 1: ITEM 2. PROPERTIES - ITEM 2. PROPERTIES
We are headquartered in Austin, Texas. Our principal facilities include a large n...
  Keyword match 2: Item 1. - Item 1.
Business
4...
  Keyword match 3: Item 1A. - Item 1A.
Risk Factors
14...
DEBUG: Final result_chunks count=5
DEBUG: Total context chars=1236
  Final chunk 1: ITEM 2. PROPERTIES (1145 chars)
  Final chunk 2: Item 1. (18 chars)
  Final chunk 3: Item 1A. (24 chars)
  Final chunk 4: Item 2. (21 chars)
  Final chunk 5: Item 3. (28 chars)
✅ Retrieved 5 relevant chunks (requested 5)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag_pipeline:Query: Where is Tesla headquartered?
INFO:rag_pipeline:  [1] tesla-2024-10K::ITEM 2. PROPERTIES::chunk-71
INFO:rag_pipeline:  [2] tesla-2024-10K::Item 1.::chunk-0
INFO:rag_pipeline:  [3] tesla-2024-10K::Item 1A.::chunk-1
INFO:rag_pipeline:  [4] tesla-2024-10K::Item 2.::chunk-2
INFO:rag_pipeline:  [5] tesla-2024-10K::Item 3.::chunk-3
INFO:rag_pipeline:Answer length: 44 chars


Answer: Tesla is headquartered in Austin, Texas [1].
Abstained: False
